# 定义 AEpT 和 Z0 介质的示例

In [ ]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import AutoMinorLocator
from numpy import log, sum
from scipy.optimize import minimize

import skrf as rf
import skrf.mathFunctions as mf
from skrf.calibration import NISTMultilineTRL

rf.stylely()

## 测量两条不同长度的CPWG传输线测量于2017年3月21日使用安立士MS46524B 20GHz VNA进行。测试设置是线性频率扫描，范围从1MHz到10GHz，数据点数为10000。输出功率为0dBm，中频带宽为1kHz，未使用平均或平滑处理。CPWGxxx是一种L长、W宽的介质基板，G为顶层接地面的间隙宽度，T为铜层厚度，H为基板高度，顶层和底层都有接地层。在传输线的两侧设置了紧密排列的通孔墙，并且通过多个通孔将顶层和底层的接地层连接起来。| 名称 | L (mm) | W (mm) | G (mm) | H (mm) | T (um) | 基板 || :--- | ---: | ---: | ---: | ---: | ---: | :--- || MSL100 | 100 | 1.70 | 0.50 | 1.55 | 50 | FR-4 || MSL200 | 200 | 1.70 | 0.50 | 1.55 | 50 | FR-4 |通过机械方式加工电路板，侧壁角度为45°。为了设计目的，假设介电常数约为4.5。![MSL100和MSL200示意图，两者均为微带线，MSL200的长度是MSL100的两倍](MSL_CPWG_100_200.jpg "MSL100和MSL200")

In [ ]:
# Load raw measurements
TL100 = rf.Network('CPWG100.s2p')
TL200 = rf.Network('CPWG200.s2p')
TL100_dc = TL100.extrapolate_to_dc(kind='linear')
TL200_dc = TL200.extrapolate_to_dc(kind='linear')

plt.figure()
plt.suptitle('Raw measurement')
TL100.plot_s_db()
TL200.plot_s_db()

plt.figure()
t0 = -2
t1 = 4
plt.suptitle('Time domain reflexion step response (DC extrapolation)')
ax = plt.subplot(1, 1, 1)
TL100_dc.plot_z_time_step(0, 0, ax=ax, color='0.0')
TL200_dc.plot_z_time_step(0, 0, ax=ax, color='0.2')
ax.set_xlim(t0, t1)
ax.xaxis.set_minor_locator(AutoMinorLocator(10))
ax.yaxis.set_minor_locator(AutoMinorLocator(5))
ax.patch.set_facecolor('1.0')
ax.grid(True, color='0.8', which='minor')
ax.grid(True, color='0.4', which='major')
plt.show()


在阶跃响应中，可以估算传输线和连接器部分的阻抗。传输线部分并非完全平坦，其阻抗存在一些变化，这可能是由制造公差和介电不均匀性引起的。请注意，在反射图中，延迟时间是有效线段延迟的两倍，因为波在传输线上来回传播。连接器不连续部分的长度约为 50 皮秒。TL100 传输线平台的长度（阻抗平坦部分）约为 450 皮秒。

In [ ]:
Z_conn = 53.2    # ohm, connector impedance
Z_line = 51.4    # ohm, line plateau impedance
d_conn = 0.05e-9 # s,   connector discontinuity delay
d_line = 0.45e-9 # s,   line plateau delay, without connectors

## 通过多线法提取介电材料的有效相对介电常数

In [ ]:
#Make the missing reflect measurement
#This is possible because we already have existing calibration
#and know what the open measurement would look like at the reference plane
#'refl_offset' needs to be set to -half_thru - connector_length.
reflect = TL100.copy()
reflect.s[:,0,0] = 1
reflect.s[:,1,1] = 1
reflect.s[:,1,0] = 0
reflect.s[:,0,1] = 0

# Perform NISTMultilineTRL algorithm. Reference plane is at the center of the thru.
cal = NISTMultilineTRL([TL100, reflect, TL200], [1], [0, 100e-3], er_est=3.0, refl_offset=[-56e-3])

plt.figure()
plt.title('Corrected lines')
cal.apply_cal(TL100).plot_s_db()
cal.apply_cal(TL200).plot_s_db()
plt.show()

校准结果显示，残余噪声水平非常低。误差模型拟合良好。

In [ ]:
from skrf.media import DefinedAEpTandZ0

freq  = TL100.frequency
f     = TL100.frequency.f
f_ghz = TL100.frequency.f/1e9
L     = 0.1
A     = 0.0
f_A   = 1e9
ep_r0 = 2.0
tanD0 = 0.001
f_ep  = 1e9
x0 = [ep_r0, tanD0]

ep_r_mea = cal.er_eff.real
A_mea    = 20/log(10)*cal.gamma.real

def model(x, freq, ep_r_mea, A_mea, f_ep):
    ep_r, tanD = x[0], x[1]
    m = DefinedAEpTandZ0(frequency=freq, ep_r=ep_r, tanD=tanD, z0=50,
                              f_low=1e3, f_high=1e18, f_ep=f_ep, model='djordjevicsvensson')
    ep_r_mod = m.ep_r_f.real
    A_mod = m.alpha * log(10)/20
    return sum((ep_r_mod - ep_r_mea)**2)  + 0.001*sum((20/log(10)*A_mod - A_mea)**2)

res = minimize(model, x0, args=(TL100.frequency, ep_r_mea, A_mea, f_ep),
               bounds=[(2, 4), (0.001, 0.013)])
ep_r, tanD = res.x[0], res.x[1]

print(f'epr={ep_r:.3f}, tand={tanD:.4f} at {f_ep * 1e-9:.1f} GHz.')

m = DefinedAEpTandZ0(frequency=freq, ep_r=ep_r, tanD=tanD, z0=50,
                              f_low=1e3, f_high=1e18, f_ep=f_ep, model='djordjevicsvensson')

plt.figure()
plt.suptitle('Effective relative permittivity and attenuation')
plt.subplot(2,1,1)
plt.ylabel(r'$\epsilon_{r,eff}$')
plt.plot(f_ghz, ep_r_mea, label='measured')
plt.plot(f_ghz, m.ep_r_f.real, label='model')
plt.legend()

plt.subplot(2,1,2)
plt.xlabel('Frequency [GHz]')
plt.ylabel('A (dB/m)')
plt.plot(f_ghz, A_mea, label='measured')
plt.plot(f_ghz, 20/log(10)*m.alpha, label='model')
plt.legend()
plt.show()


相对介电常数 $\epsilon_{e,eff}$ 和衰减 $A$ 显示出合理的吻合度。通过实施 Kirschning 和 Jansen 微带线色散模型或使用线性校正，可以获得更好的吻合效果。

## 评估连接器的影响

In [ ]:
# note: a half line is embedded in connector network
coefs = cal.coefs
r = mf.sqrt_phase_unwrap(coefs['forward reflection tracking'])
s1 = np.array([[coefs['forward directivity'],r],
        [r, coefs['forward source match']]]).transpose()

conn = TL100.copy()
conn.name = 'Connector'
conn.s = s1

# delay estimation,
phi_conn = (np.angle(conn.s[:500,1,0]))
z = np.polyfit(f[:500], phi_conn, 1)
p = np.poly1d(z)
delay = -z[0]/(2*np.pi)
print(f'Connector + half thru delay: {delay * 1e12:.0f} ps')
print(f'TDR read half thru delay: {d_line/2 * 1e12:.0f} ps')
d_conn_p = delay - d_line/2
print(f'Connector delay: {d_conn_p * 1e12:.0f} ps')


# connector model with guessed loss
half = m.line(d_line/2, 's', z0=Z_line)
mc = DefinedAEpTandZ0(m.frequency, ep_r=1, tanD=0.025, z0=50,
                              f_low=1e3, f_high=1e18, f_ep=f_ep, model='djordjevicsvensson')
left = mc.line(d_conn_p, 's', z0=Z_conn)
right = left.flipped()
check = mc.thru() ** left ** half ** mc.thru()

plt.figure()
plt.suptitle('Connector + half thru comparison')
plt.subplot(2,1,1)
conn.plot_s_deg(1, 0, label='measured')
check.plot_s_deg(1, 0, label='model')
plt.ylabel('phase (rad)')
plt.legend()

plt.subplot(2,1,2)
conn.plot_s_db(1, 0, label='Measured')
check.plot_s_db(1, 0, label='Model')
plt.xlabel('Frequency (GHz)')
plt.ylabel('Insertion Loss (dB)')
plt.legend()
plt.show()

连接器 + 直通数据的图表显示，校准结果与模型之间具有合理的吻合度。

## 最终检查

In [ ]:
DUT = m.line(d_line, 's', Z_line)
DUT.name = 'model'

Check = m.thru() ** left ** DUT ** right ** m.thru()
Check.name = 'model with connectors'

plt.figure()
TL100.plot_s_db()
Check.plot_s_db(1,0, color='k')
Check.plot_s_db(0,0, color='k')
plt.show()

Check_dc = Check.extrapolate_to_dc(kind='linear')

plt.figure()
plt.suptitle('Time domain step-response')
ax = plt.subplot(1,1,1)
TL100_dc.plot_z_time_step(0, 0, ax=ax, color='k')
Check_dc.plot_z_time_step(0, 0, ax=ax, color='b')
t0 = -2
t1 = 4
ax.set_xlim(t0, t1)
ax.xaxis.set_minor_locator(AutoMinorLocator(10))
ax.yaxis.set_minor_locator(AutoMinorLocator(5))
ax.patch.set_facecolor('1.0')
ax.grid(True, color='0.8', which='minor')
ax.grid(True, color='0.5', which='major')


图表显示，在 4 GHz 之前，模型与测量结果具有合理的吻合度。后续工作可能包括实现 CPWG 介质，或者通过增加更多线段来对线路进行建模，以考虑阻抗随位置的变化。